# Debate Results Analysis

Loads all saved debate results from `results/` and visualises scores, token usage, cost, and rule violations across runs.

Run from the repo root:
```bash
uv run jupyter notebook notebooks/results_analysis.ipynb
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = Path('..') / 'results'

debates = []
for p in sorted(RESULTS_DIR.glob('*.json')):
    with open(p) as f:
        debates.append(json.load(f))

print(f'Loaded {len(debates)} debate(s)')
for d in debates:
    print(f"  [{d['experiment_id']}] {d['topic'][:60]}...  winner={d['winner']}")

## 1. Score Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ids   = [d['experiment_id'] for d in debates]
s_pro = [d['score_pro']     for d in debates]
s_con = [d['score_con']     for d in debates]
x     = range(len(debates))

bar_w = 0.35
ax.bar([i - bar_w/2 for i in x], s_pro, bar_w, label='Pro (AXIOM)',   color='#4C72B0')
ax.bar([i + bar_w/2 for i in x], s_con, bar_w, label='Con (NEMESIS)', color='#DD8452')

ax.set_xticks(list(x))
ax.set_xticklabels(ids)
ax.set_ylabel('Judge Score')
ax.set_title('Pro vs Con Scores by Debate')
ax.set_ylim(0, 100)
ax.legend()
ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../assets/score_comparison.png', dpi=120)
plt.show()

## 2. Token Usage and Cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

tokens = [d['token_usage']['total_tokens'] for d in debates]
costs  = [d['cost_usd']                    for d in debates]

axes[0].bar(ids, tokens, color='#55A868')
axes[0].axhline(400_000, color='red', linestyle='--', linewidth=1, label='Budget (400k)')
axes[0].set_title('Total Tokens per Debate')
axes[0].set_ylabel('Tokens')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}k'))
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

axes[1].bar(ids, costs, color='#C44E52')
axes[1].set_title('Cost (USD) per Debate')
axes[1].set_ylabel('Cost ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.2f}'))
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../assets/token_cost_comparison.png', dpi=120)
plt.show()

## 3. Per-Round Token Growth

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for d in debates:
    rounds = d.get('per_round_tokens', [])
    if rounds:
        ax.plot(range(1, len(rounds) + 1), rounds, marker='o', label=d['experiment_id'])

ax.set_title('Tokens Used per Round (quadratic growth from conversation history)')
ax.set_xlabel('Round')
ax.set_ylabel('Tokens')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Summary Table

In [ ]:
print(f"{'ID':<14} {'Topic':<45} {'Winner':<6} {'Pro':>5} {'Con':>5} {'Tokens':>8} {'Cost':>7}")
print('-' * 90)
for d in debates:
    print(
        f"{d['experiment_id']:<14}"
        f" {d['topic'][:44]:<45}"
        f" {d['winner']:<6}"
        f" {d['score_pro']:>5}"
        f" {d['score_con']:>5}"
        f" {d['token_usage']['total_tokens']:>8,}"
        f" ${d['cost_usd']:>6.2f}"
    )